#Init

In [0]:
import pyspark.sql.functions as F

#Read Data From Gold Tables 

In [0]:
sales_df = spark.table("workspace.silver.crm_sales_info")
customers_df = spark.table("workspace.gold.dim_customers")
product_df = spark.table("workspace.gold.dim_product")
location_df = spark.table("workspace.gold.dim_location")

In [0]:
s = sales_df.alias("s")
c = customers_df.alias("c")
l = location_df.alias("l")
p = product_df.alias("p")

fact_df = (
    s
    .join(
        c,
        s.customer_id == c.customer_id,
        "left"
    )
    .join(
        l,
        c.location_customer_key == l.customer_key,
        "left"
    )
    .join(
        p,
        s.product_key == p.sales_product_key,
        "left"
    )
)

In [0]:
fact_df.display()

In [0]:
fact_df.columns

#Select Columns 

In [0]:
fact_df = fact_df.select(
    F.col("s.order_number"),
    F.col("s.product_key"),
    F.col("c.customer_key"),
    F.col("s.order_date"),
    F.col("s.ship_date"),
    F.col("s.due_date"),
    F.col("s.sales_amount"),
    F.col("s.quantity"),
    F.col("s.price"),
    F.col("s.is_return"),
    F.col("s.processing_days"),
    F.col("s.expected_delivery_days"),
    F.col("s.order_year"),
    F.col("s.order_month")
)

In [0]:
fact_df.count()
fact_df.groupBy("order_year").count().show()
fact_df.groupBy("is_return").count().show()

In [0]:
fact_df.display()

In [0]:
(
    fact_df.write
        .mode("overwrite")
        .option("overwriteSchema","true")
        .format("delta")
        .saveAsTable("workspace.gold.fact_sales")
)

In [0]:
gold_fact_df = spark.table("workspace.gold.fact_sales")

gold_fact_df.show()
gold_fact_df.printSchema()
gold_fact_df.count()